# Deep Past Challenge - Training v6 (Comprehensive)

**Goal**: Fine-tune mattiaangeli's byt5-akkadian-mbr-v2 (35.5) to score 40+

### Strategy
1. Download ALL available datasets (Michel letters, onomasticon, ORACC)
2. 2-phase curriculum: broad data → high-quality OA-only
3. Ultra-conservative fine-tuning (lr=5e-7, 1 epoch per phase)
4. Baseline eval BEFORE training → abort if fine-tuning hurts
5. Save best model for inference

### Requirements
- GPU: H100 (or any GPU with 40GB+ VRAM)
- Internet: ON (to download datasets)
- Datasets to attach: `mattiaangeli/byt5-akkadian-mbr-v2`

In [ ]:
# ============================================================
# Cell 1: Setup & Downloads
# ============================================================
!pip install -q sacrebleu datasets kaggle

import os, re, json, time, copy, warnings, math, gc
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import sacrebleu

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB)')

In [ ]:
# ============================================================
# Cell 2: Download additional datasets
# ============================================================
DATA_DIR = '/kaggle/working/data' if Path('/kaggle').exists() else './training_data'
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

# Download Michel Old Assyrian Letters (264 high-quality domain-matched pairs)
MICHEL_PATH = Path(DATA_DIR) / 'michel_letters.csv'
if not MICHEL_PATH.exists():
    try:
        !kaggle datasets download -d manwithacat/michel-old-assyrian-letters -p {DATA_DIR} --unzip --force
        # Find the CSV
        for f in Path(DATA_DIR).rglob('*.csv'):
            if 'michel' in f.name.lower() or 'letter' in f.name.lower():
                if f != MICHEL_PATH:
                    f.rename(MICHEL_PATH)
                break
        print(f'Michel letters downloaded: {MICHEL_PATH}')
    except Exception as e:
        print(f'Michel download failed: {e}')

# Download Old Assyrian Grammars (has onomasticon.csv)
ONOMASTICON_PATH = Path(DATA_DIR) / 'onomasticon.csv'
if not ONOMASTICON_PATH.exists():
    try:
        !kaggle datasets download -d deeppast/old-assyrian-grammars-and-other-resources -p {DATA_DIR} --unzip --force
        for f in Path(DATA_DIR).rglob('onomasticon.csv'):
            if f != ONOMASTICON_PATH:
                f.rename(ONOMASTICON_PATH)
            break
        print(f'Onomasticon downloaded: {ONOMASTICON_PATH}')
    except Exception as e:
        print(f'Onomasticon download failed: {e}')

# Check what competition data is available
COMP_DIR = '/kaggle/input/deep-past-initiative-machine-translation'
if not Path(COMP_DIR).exists():
    COMP_DIR = '../data/deep-past-initiative-machine-translation'

print(f'\nCompetition data: {COMP_DIR}')
for f in sorted(Path(COMP_DIR).glob('*.csv')):
    print(f'  {f.name}: {sum(1 for _ in open(f))-1} rows')

In [ ]:
# ============================================================
# Cell 3: Build training dataset from ALL sources
# ============================================================

PREFIX = 'translate Akkadian to English: '

# Preprocessing (minimal — match model's training)
_RE_BIG = re.compile(r'(\.{3,}|\u2026+)')
_RE_SMALL = re.compile(r'(xx+|\s+x\s+)')

def preprocess(text):
    if pd.isna(text): return ''
    text = str(text)
    text = _RE_BIG.sub('<big_gap>', text)
    text = _RE_SMALL.sub('<gap>', text)
    return text

all_pairs = []  # list of (akkadian, english, source, weight)

# --- Source 1: Competition Sentences file (highest quality) ---
sentences_path = Path(COMP_DIR) / 'Sentences_Oare_FirstWord_LinNum.csv'
if sentences_path.exists():
    sent_df = pd.read_csv(sentences_path)
    # Get sentence-level pairs that have translations
    # The sentences file has: display_name, text_uuid, sentence_uuid, translation, first_word_spelling, line_number
    # We need to match these with train.csv to get the transliterations
    train_df = pd.read_csv(Path(COMP_DIR) / 'train.csv')
    
    # Strategy: use sentences that have non-empty translations
    valid_sent = sent_df[sent_df['translation'].notna() & (sent_df['translation'].str.len() > 5)].copy()
    print(f'Sentences file: {len(valid_sent)} with translations (of {len(sent_df)} total)')
    
    # We can also use the train.csv transliterations directly
    for _, row in train_df.iterrows():
        akk = preprocess(row.get('transliteration', ''))
        eng = str(row.get('translation', ''))
        if akk and eng and len(eng) > 5:
            all_pairs.append((akk, eng, 'train_doc', 0.7))
    print(f'  train.csv docs: {len(train_df)} pairs')

# --- Source 2: Pre-processed sentence alignments (if available) ---
for strategy in ['strategy_a', 'strategy_b', 'strategy_c']:
    spath = Path('../data_processed/sentence_aligned') / f'{strategy}.csv'
    if not spath.exists():
        spath = Path('/kaggle/working/data_processed/sentence_aligned') / f'{strategy}.csv'
    if spath.exists():
        sdf = pd.read_csv(spath)
        weight = {'strategy_a': 1.0, 'strategy_b': 0.7, 'strategy_c': 0.8}[strategy]
        for _, row in sdf.iterrows():
            akk = preprocess(row.get('transliteration', row.get('akkadian', '')))
            eng = str(row.get('translation', row.get('english', '')))
            if akk and eng and len(eng) > 3:
                all_pairs.append((akk, eng, strategy, weight))
        print(f'  {strategy}: {len(sdf)} pairs (weight={weight})')

# --- Source 3: Michel Old Assyrian Letters (264 high-quality pairs) ---
if MICHEL_PATH.exists():
    michel_df = pd.read_csv(MICHEL_PATH)
    print(f'\nMichel letters: {len(michel_df)} rows, columns: {list(michel_df.columns)}')
    # Determine column names (could be akkadian/english or transliteration/translation)
    akk_col = next((c for c in michel_df.columns if c.lower() in ['akkadian', 'transliteration', 'source', 'akk']), None)
    eng_col = next((c for c in michel_df.columns if c.lower() in ['english', 'translation', 'target', 'eng']), None)
    if akk_col and eng_col:
        for _, row in michel_df.iterrows():
            akk = preprocess(str(row[akk_col]))
            eng = str(row[eng_col])
            if akk and eng and len(eng) > 3:
                all_pairs.append((akk, eng, 'michel', 1.0))  # High weight — domain match!
        print(f'  Added {len(michel_df)} Michel pairs (weight=1.0 — best domain match)')
    else:
        print(f'  Could not find akk/eng columns in Michel data. Columns: {list(michel_df.columns)}')
        print(michel_df.head())
else:
    print('Michel letters: not available')

# --- Source 4: ORACC parallel corpus (if available) ---
oracc_path = Path('../data_processed/oracc/oracc_parallel.csv')
if not oracc_path.exists():
    oracc_path = Path('/kaggle/working/oracc_parallel.csv')
if oracc_path.exists():
    oracc_df = pd.read_csv(oracc_path)
    for _, row in oracc_df.iterrows():
        akk = preprocess(row.get('akkadian', ''))
        eng = str(row.get('english', ''))
        if akk and eng and len(eng) > 5:
            all_pairs.append((akk, eng, 'oracc', 0.4))  # Lower weight — domain mismatch
    print(f'  ORACC: {len(oracc_df)} pairs (weight=0.4 — domain mismatch)')

# --- Source 5: Combined train_split (if pre-processed data exists) ---
combined_path = Path('../data_processed/combined/train_split.csv')
if combined_path.exists() and len(all_pairs) < 500:  # Only use if we didn't get enough from above
    cdf = pd.read_csv(combined_path)
    for _, row in cdf.iterrows():
        src = row.get('source', 'unknown')
        if src == 'lexicon': continue  # Skip lexicon (dictionary defs, not translations)
        akk = preprocess(row.get('transliteration', ''))
        eng = str(row.get('translation', ''))
        w = row.get('weight', 0.5)
        if akk and eng and len(eng) > 3:
            all_pairs.append((akk, eng, src, w))
    print(f'  Combined train_split: {len(cdf)} rows')

# Deduplicate by (akkadian, english) pair
seen = set()
unique_pairs = []
for akk, eng, src, w in all_pairs:
    key = (akk.strip()[:200], eng.strip()[:200])  # Use prefix for dedup
    if key not in seen:
        unique_pairs.append((akk, eng, src, w))
        seen.add(key)

train_data = pd.DataFrame(unique_pairs, columns=['akkadian', 'english', 'source', 'weight'])

print(f'\n=== Training Data Summary ===')
print(f'Total unique pairs: {len(train_data)}')
print(f'By source:')
for src, grp in train_data.groupby('source'):
    print(f'  {src}: {len(grp)} pairs (weight={grp["weight"].mean():.2f})')

In [ ]:
# ============================================================
# Cell 4: Load model & tokenizer
# ============================================================

MODEL_PATHS = [
    '/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1',
    '/kaggle/input/byt5-akkadian-mbr-v2/pytorch/default/1',
    '/kaggle/input/byt5-akkadian-mbr-v2',
    '../models/byt5-akkadian/final',
]
MODEL_PATH = None
for p in MODEL_PATHS:
    if Path(p).exists():
        MODEL_PATH = p
        break
assert MODEL_PATH, 'No base model found!'

print(f'Loading base model: {MODEL_PATH}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded: {n_params/1e6:.0f}M params')

# Save base weights for rollback
base_weights = copy.deepcopy(model.state_dict())
print('Base weights saved for potential rollback')

In [ ]:
# ============================================================
# Cell 5: Tokenize dataset
# ============================================================

class AkkadianDataset(Dataset):
    def __init__(self, df, tokenizer, max_src=384, max_tgt=256):
        self.examples = []
        for _, row in df.iterrows():
            src = PREFIX + str(row['akkadian'])
            tgt = str(row['english'])
            src_tok = tokenizer(src, max_length=max_src, truncation=True, padding=False)
            with tokenizer.as_target_tokenizer():
                tgt_tok = tokenizer(tgt, max_length=max_tgt, truncation=True, padding=False)
            self.examples.append({
                'input_ids': src_tok['input_ids'],
                'attention_mask': src_tok['attention_mask'],
                'labels': tgt_tok['input_ids'],
            })
    
    def __len__(self): return len(self.examples)
    def __getitem__(self, i): return self.examples[i]

# Split: use high-quality data for training, reserve some for eval
# Prioritize: michel (1.0), strategy_a (1.0), train_doc (0.7)
high_q = train_data[train_data['weight'] >= 0.7].copy()
low_q = train_data[train_data['weight'] < 0.7].copy()

# Use 90% of high-quality for training, 10% for eval
eval_size = min(100, max(20, len(high_q) // 10))
eval_df = high_q.sample(n=eval_size, random_state=42)
train_hq = high_q.drop(eval_df.index)

print(f'High-quality train: {len(train_hq)}')
print(f'Low-quality (broad): {len(low_q)}')
print(f'Eval: {eval_size}')

# Phase 1: ALL data (broad learning)
phase1_df = pd.concat([train_hq, low_q], ignore_index=True)
# Phase 2: High-quality only (domain specialization)
phase2_df = train_hq.copy()

phase1_ds = AkkadianDataset(phase1_df, tokenizer)
phase2_ds = AkkadianDataset(phase2_df, tokenizer)
eval_ds = AkkadianDataset(eval_df, tokenizer)

print(f'\nPhase 1 dataset: {len(phase1_ds)} examples')
print(f'Phase 2 dataset: {len(phase2_ds)} examples')
print(f'Eval dataset: {len(eval_ds)} examples')

In [ ]:
# ============================================================
# Cell 6: Evaluation function
# ============================================================
_CHRF = sacrebleu.metrics.CHRF(word_order=2)
_BLEU = sacrebleu.metrics.BLEU(effective_order=True)

def evaluate(model, eval_df, tokenizer, device, max_samples=80, num_beams=5):
    """Quick evaluation: compute GeoMean of BLEU and chrF++ on eval set."""
    model.eval()
    sample = eval_df.head(max_samples)
    
    preds, refs = [], []
    for _, row in sample.iterrows():
        src = PREFIX + str(row['akkadian'])
        ref = str(row['english'])
        
        inputs = tokenizer(src, return_tensors='pt', max_length=384, truncation=True).to(device)
        with torch.inference_mode():
            out = model.generate(
                **inputs, num_beams=num_beams, max_new_tokens=256,
                length_penalty=1.3, repetition_penalty=1.2,
                early_stopping=True,
            )
        pred = tokenizer.decode(out[0], skip_special_tokens=True)
        preds.append(pred)
        refs.append(ref)
    
    bleu = _BLEU.corpus_score(preds, [refs]).score
    chrf = _CHRF.corpus_score(preds, [refs]).score
    geo = math.sqrt(max(bleu, 0.1) * max(chrf, 0.1))
    
    return {'bleu': bleu, 'chrf': chrf, 'geomean': geo, 'n_samples': len(preds)}

# Baseline evaluation BEFORE any training
print('Evaluating BASE model (before fine-tuning)...')
base_scores = evaluate(model, eval_df, tokenizer, device)
print(f'BASE MODEL: BLEU={base_scores["bleu"]:.2f}, chrF++={base_scores["chrf"]:.2f}, GeoMean={base_scores["geomean"]:.2f}')
print(f'  (on {base_scores["n_samples"]} eval samples)')

best_geomean = base_scores['geomean']
best_phase = 'base'

In [ ]:
# ============================================================
# Cell 7: Training (ultra-conservative, 2-phase)
# ============================================================

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

PHASES = [
    {
        'name': 'Phase 1: Broad',
        'dataset': phase1_ds,
        'lr': 5e-7,           # Ultra-conservative
        'epochs': 1,
        'warmup': 50,
        'label_smoothing': 0.0,  # No smoothing — preserve model knowledge
    },
    {
        'name': 'Phase 2: High-Quality OA',
        'dataset': phase2_ds,
        'lr': 2e-7,           # Even lower for domain specialization
        'epochs': 1,
        'warmup': 30,
        'label_smoothing': 0.0,
    },
]

results_table = [{'phase': 'Base', 'bleu': base_scores['bleu'], 'chrf': base_scores['chrf'], 'geomean': base_scores['geomean']}]

for phase_cfg in PHASES:
    print(f'\n{"="*60}')
    print(f'{phase_cfg["name"]} ({len(phase_cfg["dataset"])} samples, lr={phase_cfg["lr"]})')
    print(f'{"="*60}')
    
    output_dir = Path(OUTPUT_DIR if Path('/kaggle').exists() else './output') / phase_cfg['name'].replace(' ', '_').replace(':', '')
    
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=phase_cfg['epochs'],
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,  # effective batch = 32
        learning_rate=phase_cfg['lr'],
        warmup_steps=phase_cfg['warmup'],
        weight_decay=0.01,
        label_smoothing_factor=phase_cfg['label_smoothing'],
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        save_strategy='no',
        predict_with_generate=False,
        dataloader_num_workers=2,
        remove_unused_columns=False,
    )
    
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=phase_cfg['dataset'],
        data_collator=data_collator,
        tokenizer=tokenizer,
    )
    
    trainer.train()
    
    # Evaluate after this phase
    print(f'Evaluating after {phase_cfg["name"]}...')
    scores = evaluate(model, eval_df, tokenizer, device)
    results_table.append({
        'phase': phase_cfg['name'],
        'bleu': scores['bleu'],
        'chrf': scores['chrf'],
        'geomean': scores['geomean'],
    })
    print(f'  BLEU={scores["bleu"]:.2f}, chrF++={scores["chrf"]:.2f}, GeoMean={scores["geomean"]:.2f}')
    
    if scores['geomean'] > best_geomean:
        best_geomean = scores['geomean']
        best_phase = phase_cfg['name']
        # Save best weights
        best_weights = copy.deepcopy(model.state_dict())
        print(f'  NEW BEST! (GeoMean {scores["geomean"]:.2f} > previous {best_geomean:.2f})')
    else:
        print(f'  No improvement (GeoMean {scores["geomean"]:.2f} <= best {best_geomean:.2f})')
        # Restore best weights and STOP
        if best_phase == 'base':
            print(f'  ABORTING: Fine-tuning hurt the model. Restoring base weights.')
            model.load_state_dict(base_weights)
            break
        else:
            print(f'  Restoring weights from {best_phase}')
            model.load_state_dict(best_weights)
            break

print(f'\n=== Results ===')
for r in results_table:
    marker = ' ← BEST' if r['phase'] in [best_phase, 'Base' if best_phase == 'base' else ''] else ''
    print(f'  {r["phase"]:30s} BLEU={r["bleu"]:6.2f}  chrF++={r["chrf"]:6.2f}  GeoMean={r["geomean"]:6.2f}{marker}')

In [ ]:
# ============================================================
# Cell 8: Save best model
# ============================================================

SAVE_DIR = '/kaggle/working/byt5-akkadian-finetuned' if Path('/kaggle').exists() else './models/byt5-akkadian-finetuned'
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

# Save whichever is best
if best_phase == 'base':
    print('Saving BASE model (fine-tuning did not improve it)')
    model.load_state_dict(base_weights)
else:
    print(f'Saving FINE-TUNED model from {best_phase}')

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save training info
info = {
    'best_phase': best_phase,
    'best_geomean': best_geomean,
    'results': results_table,
    'base_model': MODEL_PATH,
    'training_data_size': len(train_data),
}
with open(Path(SAVE_DIR) / 'training_info.json', 'w') as f:
    json.dump(info, f, indent=2)

print(f'Saved to: {SAVE_DIR}')
print(f'Best phase: {best_phase}, GeoMean: {best_geomean:.2f}')
print(f'\nFiles:')
for f in sorted(Path(SAVE_DIR).glob('*')):
    print(f'  {f.name} ({f.stat().st_size/1e6:.1f}MB)')